In [4]:
import pandas as pd
import numpy as np

DATA_PATH = r"/media/user/DATA21/Kajeeth/covariate_anomaly detection/dataset/lead_406_buildings_cleaned.csv"

df = pd.read_csv(DATA_PATH)

remove_bid = [32, 534, 558, 653, 693, 723, 739,
              855, 910, 970, 1147, 1183, 1264, 1282]

df = df[~df["building_id"].isin(remove_bid)]

df1 = pd.read_csv("/media/user/DATA21/Kajeeth/covariate_anomaly detection/dataset/train_features.csv")

valid_buildings = df1["building_id"].unique()

df = df[df["building_id"].isin(valid_buildings)]
df = df[df["building_id"] > 263]

print("Buildings:", df.building_id.nunique())
print("Rows:", len(df))

Buildings: 153
Rows: 1343952


In [5]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

summary = []

for c in numeric_cols:
    summary.append({
        "column": c,
        "nan_count": df[c].isna().sum(),
        "inf_count": np.isinf(df[c]).sum()
    })

pd.DataFrame(summary)

,column,nan_count,inf_count
0,building_id,0,0
1,meter_reading,38354,0
2,anomaly,0,0


In [6]:
results = []

for bid in sorted(df.building_id.unique()):

    bdf = df[df.building_id == bid]

    meter = bdf["meter_reading"].values

    results.append({
        "building_id": bid,
        "rows": len(bdf),
        "nan_meter": np.isnan(meter).sum(),
        "inf_meter": np.isinf(meter).sum(),
        "std": np.nanstd(meter),
        "min": np.nanmin(meter),
        "max": np.nanmax(meter),
        "unique_values": len(np.unique(meter))
    })

diag = pd.DataFrame(results)

diag.sort_values("rows").head(20)

,building_id,rows,nan_meter,inf_meter,std,min,max,unique_values
0,270,8784,47,0,10.298389,0.000,79.080,2648
97,1074,8784,0,0,31.246246,18.508,153.058,7953
98,1106,8784,0,0,56.122417,0.000,322.000,3071
99,1120,8784,13,0,24.795660,0.000,161.586,8268
100,1128,8784,235,0,16.168837,0.000,436.751,7596
101,1137,8784,1,0,10.386385,0.000,55.000,53
102,1141,8784,1,0,5.813544,1.000,41.000,37
103,1143,8784,0,0,29.874288,25.000,187.000,135
104,1172,8784,3,0,69.259990,1.000,373.935,8412
105,1219,8784,1,0,52.276186,1.000,374.000,200


In [7]:
suspicious = diag[
    (diag["rows"] < 500) |
    (diag["nan_meter"] > 0) |
    (diag["inf_meter"] > 0) |
    (diag["unique_values"] < 10) |
    (diag["std"] == 0)
]

suspicious.sort_values("building_id")

,building_id,rows,nan_meter,inf_meter,std,min,max,unique_values
0,270,8784,47,0,10.298389,0.0,79.080,2648
1,275,8784,7,0,38.253042,0.0,261.240,6401
2,276,8784,7,0,44.546050,0.0,336.590,6728
3,278,8784,250,0,77.710721,0.0,429.740,6816
6,312,8784,24,0,4.223228,0.0,25.640,524
...,...,...,...,...,...,...,...,...
147,1315,8784,29,0,34.078098,1.0,175.500,589
148,1316,8784,39,0,22.570737,1.0,119.125,2482
149,1318,8784,67,0,79.287563,0.0,411.040,3794
150,1319,8784,1203,0,40.712549,1.0,390.410,2480


In [9]:
issues = []

for bid in sorted(df.building_id.unique()):

    bdf = (
        df[df.building_id == bid]
        .sort_values("timestamp")
    )

    ts = pd.to_datetime(bdf["timestamp"])

    duplicated = ts.duplicated().sum()

    issues.append({
        "building_id": bid,
        "duplicates": duplicated,
        "rows": len(ts)
    })

pd.DataFrame(issues).query("duplicates > 0")

,building_id,duplicates,rows


In [10]:
buildings = sorted(df.building_id.unique())

for i, bid in enumerate(buildings):
    print(i, bid)

0 270
1 275
2 276
3 278
4 290
5 293
6 312
7 318
8 335
9 345
10 356
11 423
12 439
13 492
14 560
15 623
16 657
17 658
18 666
19 667
20 673
21 675
22 677
23 680
24 683
25 685
26 687
27 697
28 698
29 701
30 708
31 710
32 721
33 722
34 729
35 730
36 732
37 742
38 801
39 827
40 844
41 848
42 879
43 880
44 881
45 882
46 884
47 886
48 887
49 889
50 890
51 892
52 893
53 894
54 895
55 896
56 903
57 905
58 909
59 914
60 919
61 922
62 924
63 925
64 926
65 928
66 929
67 931
68 935
69 936
70 941
71 942
72 945
73 948
74 950
75 952
76 961
77 966
78 967
79 968
80 969
81 971
82 973
83 974
84 975
85 977
86 978
87 981
88 988
89 990
90 992
91 994
92 996
93 1001
94 1007
95 1068
96 1073
97 1074
98 1106
99 1120
100 1128
101 1137
102 1141
103 1143
104 1172
105 1219
106 1225
107 1226
108 1230
109 1232
110 1234
111 1238
112 1239
113 1241
114 1242
115 1245
116 1246
117 1247
118 1249
119 1251
120 1252
121 1253
122 1255
123 1257
124 1258
125 1259
126 1260
127 1261
128 1266
129 1267
130 1272
131 1275
132 1278
133 12